# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [13]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [14]:
import os
import pandas as pd
import tiktoken
from pydantic import BaseModel, Field
from openai import OpenAI
from langchain_community.document_loaders import PyPDFLoader

from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# The API key is read from the environment (loaded from the .secrets file above)
client = OpenAI()

GEN_MODEL = "gpt-4o-mini"     
JUDGE_MODEL = "gpt-4o-mini"   
TONE = "Victorian English"
MAX_SUMMARY_TOKENS = 1000

/var/folders/db/hpfj4gsd2gj7nffmyz4y0c380000gn/T/ipykernel_47709/2382584789.py:9: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [15]:
from langchain_community.document_loaders import PyPDFLoader

In [16]:
PDF_PATH = "documents/managing_oneself.pdf"

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Pages: {len(docs)} | Characters: {len(document_text):,}")
print(document_text[:500])

Pages: 13 | Characters: 51,452
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [24]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",                          # keep the quotes
    api_key="not-used",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},    # confirm the header name in the course examples
)

GEN_MODEL = "gpt-4o-mini"
TONE = "Victorian English"
MAX_SUMMARY_TOKENS = 1000

print("Gateway key loaded:", bool(os.getenv("API_GATEWAY_KEY")))

Gateway key loaded: True


In [25]:
from pydantic import BaseModel, Field

# Fields the model writes
class SummaryLLM(BaseModel):
    Author: str = Field(description="Author of the article")
    Title: str = Field(description="Title of the article")
    Relevance: str = Field(description="One paragraph at most: why this article is relevant for an AI professional's development")
    Summary: str = Field(description=f"Concise summary, at most {MAX_SUMMARY_TOKENS} tokens, in the requested tone")
    Tone: str = Field(description="The tone used to write the summary")

# Final object: model fields + token counts taken from the response object
class SummaryOutput(SummaryLLM):
    InputTokens: int
    OutputTokens: int

In [26]:
# Developer prompt: fixed instructions with placeholders (no document inside)
DEVELOPER_PROMPT_TEMPLATE = """You are an expert editor who summarizes articles for professionals.

Rules:
- Extract the author and title of the article.
- Write a 'Relevance' statement (ONE paragraph at most) explaining why the article is relevant to an AI professional's professional development.
- Write a concise, faithful summary of at most {max_tokens} tokens. Do not invent facts that are not in the article.
- Write the summary in this tone: {tone}. Keep the tone consistent from the first sentence to the last, while keeping the content clear.
- Report the tone you used in the 'Tone' field.
- Do not include harmful, offensive or misleading content."""

# User prompt: the context, added dynamically at runtime
USER_PROMPT_TEMPLATE = """Summarize the following article.

<article>
{document}
</article>"""

def build_prompts(document, tone=TONE, max_tokens=MAX_SUMMARY_TOKENS):
    developer = DEVELOPER_PROMPT_TEMPLATE.format(tone=tone, max_tokens=max_tokens)
    user = USER_PROMPT_TEMPLATE.format(document=document)
    return developer, user

In [27]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")   # tokenizer for the gpt-4o family

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def generate_summary(developer_prompt: str, user_prompt: str) -> SummaryOutput:
    response = client.chat.completions.parse(
        model=GEN_MODEL,
        messages=[
            {"role": "developer", "content": developer_prompt},  # if the gateway rejects "developer", use "system"
            {"role": "user", "content": user_prompt},
        ],
        response_format=SummaryLLM,     # structured output validated against the Pydantic model
        temperature=0.3,
    )
    result = SummaryOutput(
        **response.choices[0].message.parsed.model_dump(),
        InputTokens=response.usage.prompt_tokens,        # from the response object
        OutputTokens=response.usage.completion_tokens,   # from the response object
    )
    # Pydantic can't enforce a token limit, so check it here
    n = count_tokens(result.Summary)
    if n > MAX_SUMMARY_TOKENS:
        print(f"WARNING: summary is {n} tokens (> {MAX_SUMMARY_TOKENS})")
    return result

In [28]:
dev_prompt, usr_prompt = build_prompts(document_text)
summary_v1 = generate_summary(dev_prompt, usr_prompt)

print(summary_v1.model_dump_json(indent=2))
print("Summary tokens:", count_tokens(summary_v1.Summary))

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "This article is profoundly relevant to AI professionals as it emphasizes the necessity of self-awareness and self-management in an era where knowledge workers must navigate their own careers. Understanding one's strengths, values, and preferred working styles is crucial for maximizing productivity and achieving excellence in the rapidly evolving field of artificial intelligence.",
  "Summary": "In this age of unparalleled opportunity, the esteemed Peter F. Drucker elucidates the imperative for individuals to take charge of their own professional destinies, particularly in the realm of knowledge work. He posits that success is contingent upon a profound understanding of oneself—one's strengths, weaknesses, learning styles, and values. The article delineates a series of introspective inquiries that one must undertake to cultivate this self-awareness, such as identifying personal strengths through feedback an

### Decisions
- **Model:** gpt-4o-mini (not in the GPT-5 family), cheap and supports structured outputs.
- **Tone:** Victorian English, very distinctive, so it is easy to identify and to evaluate later.
- **Prompts:** the developer prompt holds fixed instructions with `{tone}` and `{max_tokens}` placeholders; the user prompt holds the article, inserted with `str.format`, so nothing is hard-coded.
- **Token counts:** `InputTokens` and `OutputTokens` come from `response.usage`, not from the model, since LLMs can't count their own tokens reliably.
- **Summary length:** Pydantic can't enforce a token limit, so I ask for it in the instructions and verify it with `tiktoken`.

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [29]:
from deepeval.models import DeepEvalBaseLLM

JUDGE_MODEL = "gpt-4o-mini"

class GatewayJudge(DeepEvalBaseLLM):
    """Lets DeepEval use the course gateway client as its judge model."""
    def __init__(self, model_name=JUDGE_MODEL):
        self.model_name = model_name

    def load_model(self):
        return client

    def generate(self, prompt: str, schema=None):
        if schema is not None:
            resp = client.chat.completions.parse(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                response_format=schema,
            )
            return resp.choices[0].message.parsed
        resp = client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
        )
        return resp.choices[0].message.content

    async def a_generate(self, prompt: str, schema=None):
        return self.generate(prompt, schema)

    def get_model_name(self):
        return self.model_name

judge = GatewayJudge()

In [30]:
# Summarization metric: bespoke yes/no questions about the article's content
SUMMARIZATION_QUESTIONS = [
    "Does the summary state that knowing one's strengths (e.g., through feedback analysis) is the basis for self-management?",
    "Does the summary mention that people should understand how they perform and learn (e.g., reader vs. listener)?",
    "Does the summary mention that individuals need to know their values and that these must fit their organization?",
    "Does the summary convey that knowledge workers must take responsibility for managing their own careers?",
    "Does the summary cover the idea of planning for the second half of life?",
]

COHERENCE_STEPS = [
    "Does the text follow a logical order, with ideas that connect from one sentence to the next?",
    "Is each sentence clear and understandable on its own?",
    "Are transitions between ideas smooth rather than abrupt?",
    "Is the text free of contradictions and repeated or redundant statements?",
    "Does the text read as one unified piece rather than a list of unrelated points?",
]

def tonality_steps(tone):
    return [
        f"Does the vocabulary clearly belong to the target tone ({tone})?",
        f"Does the sentence structure and syntax resemble {tone}?",
        f"Is the {tone} tone maintained consistently from the first sentence to the last?",
        "Does the tone remain readable and professional, i.e., does the style avoid obscuring the meaning?",
        f"Would a typical reader immediately recognize the text as being written in {tone}?",
    ]

SAFETY_STEPS = [
    "Is the text free of offensive, hateful, or discriminatory language?",
    "Is the text free of content that encourages violence, self-harm, or illegal activity?",
    "Is the text free of personal data or sensitive private information about individuals?",
    "Does the text avoid misleading or fabricated claims presented as fact?",
    "Is the text appropriate for a general professional audience?",
]

In [31]:
from pydantic import BaseModel
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# Structured output: one score and one reason per metric
class EvaluationResult(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

def evaluate_summary(document: str, summary, tone: str = TONE) -> dict:
    test_case = LLMTestCase(input=document, actual_output=summary.Summary)

    def make_geval(name, steps):
        return GEval(
            name=name,
            evaluation_steps=steps,
            evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
            model=judge,
        )

    metrics = {
        "Summarization": SummarizationMetric(
            threshold=0.5,
            model=judge,
            assessment_questions=SUMMARIZATION_QUESTIONS,
            include_reason=True,
        ),
        "Coherence": make_geval("Coherence", COHERENCE_STEPS),
        "Tonality": make_geval("Tonality", tonality_steps(tone)),
        "Safety": make_geval("Safety", SAFETY_STEPS),
    }

    results = {}
    for name, metric in metrics.items():
        metric.measure(test_case)
        results[f"{name}Score"] = metric.score
        results[f"{name}Reason"] = metric.reason

    return EvaluationResult(**results).model_dump()   # validated, flat key-value output

/var/folders/db/hpfj4gsd2gj7nffmyz4y0c380000gn/T/ipykernel_47709/600117117.py:3: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [33]:
eval_v1 = evaluate_summary(document_text, summary_v1)
for k, v in eval_v1.items():
    print(f"{k}: {v}\n")

Output()

Output()

Output()

Output()

SummarizationScore: 0.7333333333333333

SummarizationReason: The score is 0.73 because the summary includes several pieces of extra information that are not found in the original text, leading to a lack of alignment with the original message. There are no contradictions present, which maintains some accuracy, but the addition of new concepts detracts from the clarity and focus of the summary.

CoherenceScore: 0.8

CoherenceReason: The response follows a logical order, presenting a coherent narrative that effectively connects Drucker's ideas about self-management and knowledge work. Each sentence is clear and contributes meaningfully to the overall argument without redundancy. Transitions between the concepts are smooth, maintaining a unified tone throughout. However, while the text is coherent, it could benefit from more explicit transitions to further enhance flow between some ideas, particularly when moving from personal insights to organizational alignment.

TonalityScore: 0.3

Tona

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [34]:
# User prompt for the second pass: carries the article, the first summary and the judge's feedback
ENHANCEMENT_PROMPT_TEMPLATE = """Improve the summary of the article below.

<article>
{document}
</article>

<previous_summary>
{previous_summary}
</previous_summary>

<evaluation>
{evaluation}
</evaluation>

Produce a new, improved summary. Specifically:
- Address every weakness mentioned in the evaluation reasons.
- Remove any claim or concept that is not stated in the article, and keep only what the article supports.
- If the tone score is below 0.7, make the {tone} style markedly stronger and more recognizable (vocabulary, syntax, sentence rhythm) while keeping the content clear and accurate.
- Keep the strengths that were already working.
- Stay under {max_tokens} tokens."""

def format_evaluation(ev: dict) -> str:
    names = ["Summarization", "Coherence", "Tonality", "Safety"]
    return "\n".join(
        f"- {n}: score={ev[n + 'Score']:.2f}. Reason: {ev[n + 'Reason']}" for n in names
    )

def build_enhancement_prompt(document, previous, ev, tone=TONE, max_tokens=MAX_SUMMARY_TOKENS):
    return ENHANCEMENT_PROMPT_TEMPLATE.format(
        document=document,
        previous_summary=previous.Summary,
        evaluation=format_evaluation(ev),
        tone=tone,
        max_tokens=max_tokens,
    )

In [35]:
# Same developer (instructions) prompt as before; only the user prompt changes
enh_prompt = build_enhancement_prompt(document_text, summary_v1, eval_v1)
summary_v2 = generate_summary(dev_prompt, enh_prompt)

print(summary_v2.model_dump_json(indent=2))
print("Summary tokens:", count_tokens(summary_v2.Summary))

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "This article is profoundly relevant to AI professionals as it emphasizes the necessity of self-awareness and self-management in the rapidly evolving landscape of knowledge work. As AI continues to transform industries, understanding one's strengths, values, and preferred working styles becomes imperative for navigating career trajectories and maximizing contributions to organizations.",
  "Summary": "In this epoch of unparalleled opportunity, the venerable Peter F. Drucker expounds upon the paramount importance of self-management for the knowledge worker. He asserts that success is predicated upon a profound comprehension of oneself—one's strengths, weaknesses, learning modalities, and intrinsic values. To cultivate such self-awareness, Drucker posits a series of introspective inquiries: What are my strengths? How do I perform? What are my values? Where do I belong? What can I contribute? By engaging in th

In [36]:
eval_v2 = evaluate_summary(document_text, summary_v2)
for k, v in eval_v2.items():
    print(f"{k}: {v}\n")

Output()

Output()

Output()

Output()

SummarizationScore: 1.0

SummarizationReason: The score is 1.00 because the summary accurately reflects the content of the original text without any contradictions or omissions, demonstrating a perfect alignment in information.

CoherenceScore: 1.0

CoherenceReason: The text follows a logical structure with coherent sentences that connect well, maintaining clarity throughout. Transitions between ideas flow smoothly, and there are no contradictions or redundancies present. Overall, it reads as a unified argument rather than a disjointed collection of statements, strongly adhering to the evaluation steps.

TonalityScore: 0.8

TonalityReason: The vocabulary used consistently reflects Victorian English with terms like 'venerable' and 'epoch of unparalleled opportunity.' The sentence structure is complex and formal, characteristic of the era, maintaining a proper linguistic flow. The tone is uniform throughout, avoiding modern colloquialisms, which reinforces its Victorian essence. However,

In [38]:

import pandas as pd

names = ["Summarization", "Coherence", "Tonality", "Safety"]
comparison = pd.DataFrame({
    "Original": [eval_v1[f"{n}Score"] for n in names],
    "Enhanced": [eval_v2[f"{n}Score"] for n in names],
}, index=names)
comparison["Change"] = comparison["Enhanced"] - comparison["Original"]
comparison.round(2)

,Original,Enhanced,Change
Summarization,0.73,1.0,0.27
Coherence,0.80,1.0,0.20
Tonality,0.30,0.8,0.50
Safety,1.00,0.8,-0.20


## Results and reflection

| Metric | Original | Enhanced | Change |
|---|---|---|---|
| Summarization | 0.73 | 1.00 | +0.27 |
| Coherence | 0.80 | 1.00 | +0.20 |
| Tonality | 0.30 | 0.80 | +0.50 |
| Safety | 1.00 | 0.80 | -0.20 |

**Did I get a better output?** Yes, mostly. Three of the four scores became higher. The biggest change is Tonality, from 0.30 to 0.80. The first summary sounded like normal modern English. The second summary sounds clearly Victorian. Summarization went up to 1.00, so the judge did not find extra information that is not in the article. Coherence also went up to 1.00. But Safety went down from 1.00 to 0.80. [Add one sentence here about what the judge said in SafetyReason. My guess is that the stronger Victorian style made some words less neutral, but I need to check the real reason before I say this.]

**Why did it get better?** The second prompt was not a new start. It gave the model the article, the first summary, and the comments from the judge. The comments said that the summary had extra ideas and that the tone was not Victorian enough. With this information, the model could fix the draft. It removed the ideas that are not in the article and made the style stronger. I also wrote in the prompt that the tone must be stronger if its score is below 0.7. I think this is the reason Tonality changed so much. But I ran it only one time, and the model and the judge can give different results each time. So I should not trust a big change too much.

**A trade-off:** When one score gets better, another score can get worse. Here, the stronger tone may have reduced Safety. This shows that the metrics are connected, and fixing one metric alone can make a new problem.

**Are these controls enough? No.**
- **The judge is not perfect.** The judge and the generator are from the same model family, so they can have the same weak points. A different or stronger judge and a human check would be better.
- **The model can learn the test.** In the loop, the judge's comments go back to the generator. So the summary can become good for the metrics but not really better (Goodhart's law). A separate set of new questions would be a fairer final test.
- **A score of 1.00 does not mean no mistakes.** The summary can still have hallucinations. Checking each claim against the article would be stronger.
- **Some rules are better checked with code.** The 1000-token limit, the one-paragraph relevance, and the output format are checked by tiktoken and Pydantic, not by the LLM.
- **One article is not enough.** I used one article and one run. A real system needs more documents, repeated runs with average scores, a stop rule for the loop (for example, a maximum number of rounds or a target score), and monitoring after deployment.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
